# 4.0 本章表格的讀法

全章各表共用同一組欄位。先一次講清楚，後續不再重複。

| 欄位 | 意義 | 如何得到 |
| :--- | :--- | :--- |
| **pp**（百分點） | **兩個百分比相減**的單位。年化 1.5% 對 0.7% 的差是 **0.8 pp**，不是 0.8% | 差值本身 |
| **年化Δ** | 處理組減對照組的**年化報酬差**，正值代表處理組較優 | 逐日差分 $\Delta r_t$ 的平均 × 252 |
| **95% CI** | 效果量的信賴區間 | bootstrap 分布的 2.5／97.5 百分位（§3.5） |
| **$p$** | 雙尾 $p$ 值 | 將 bootstrap 分布平移至中心為零後，觀測值的極端程度 |
| **BH 校正 $p$** | 多重比較校正後的 $p$ | Benjamini-Hochberg 控制 FDR；同一項檢定的各組一起校正 |
| **IR**（資訊比率） | 差分序列的**風險調整後**效果 | 年化Δ ÷ 差分序列的年化標準差 |
| **勝日%** | 處理組當日報酬高於對照組的交易日占比 | $\#\{\Delta r_t > 0\}$ ÷ 交易日數 |
| **逐格顯著** | 15 個參數格中，**單獨**檢定即達 5% 顯著者的格數 | 每格各跑一次 bootstrap |

::: {.callout-important}

### 兩個最容易誤讀的欄位

**「pp」不是「%」。** 本研究的策略絕對年化報酬多落在 −0.6% 至 +0.9%，
而交易層的效果量中位為 +0.579 **pp**——**效果量與報酬水準本身同量級**。
這不矛盾：Δ 是兩臂相減，共同的市場成分已被消去（§3.5）。

**「逐格顯著 2/15」不是「只有 2 格有效」。** 15 格方向多數一致卻只有少數
單獨顯著，代表的是**單格樣本不足以偵測**，而非效果只出現在那幾格。
主張一律以等權組合為準（§3.5 報告口徑）。

:::

::: {.aside}
另有 $SE$（標準誤）與 MDE（最小可偵測效果）兩個量出現於 §4.1.3 與 §4.5，
其定義與推導於該處給出。
:::


# 4.1 分組層：限制強度與配對品質

## 主檢定：9 組對照，方向全部偏向 GICS

固定排序準則與交易端，唯一變因為分組方法。$\Delta = r_{ML} - r_{GICS}$，正值代表資料驅動分群較優。

| 分群 | 排序 | 年化Δ(pp) | 95% CI (pp) | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | :---: | ---: | ---: |
| Agglomerative | DTW | −0.164 | [−0.98, +0.62] | 0.689 | 0.689 |
| HDBSCAN | DTW | −0.289 | [−1.04, +0.44] | 0.454 | 0.531 |
| Agglomerative | SDP | −0.302 | [−1.11, +0.46] | 0.458 | 0.531 |
| Agglomerative | SSD | −0.336 | [−1.30, +0.52] | 0.472 | 0.531 |
| HDBSCAN | SDP | −0.538 | [−1.29, +0.15] | 0.139 | 0.250 |
| HDBSCAN | SSD | −0.664 | [−1.52, +0.07] | 0.097 | 0.218 |
| K-means | SSD | −0.881 | [−1.86, +0.03] | 0.066 | 0.218 |
| K-means | DTW | −0.922 | [−1.98, +0.12] | 0.083 | 0.218 |
| K-means | SDP | **−0.996** | [−2.04, **−0.07**] | **0.048** | 0.218 |

**方向 9/9 偏向 GICS，BH 校正後無一顯著**（校正後最小 $p$ = 0.218）。
校正前有一組達 5% 顯著——K-means × SDP，區間整段落在零的左側，
**方向是 GICS 較優**，即該格指向資料驅動分群顯著較差。

> 九組方向全部一致，此型態比任何單組的 $p$ 值更有訊息量。
> 惟九組彼此不獨立（每三組共用同一 GICS 臂），故不施加正式的方向一致性檢定。

:::  {.aside}
全程並行計算 Newey-West HAC 作為參數法對照，**17 組對照兩法的方向與量級一致**；分組層與兩層組合在兩法下皆無顯著，交易層兩法皆有四組於校正前顯著。完整雙欄見 `results/analysis/` 各 CSV。
:::

## 兩種「不顯著」的性質不同

::: {.callout-important}

雙尾檢定不顯著 **不等於** 兩者相當。區分兩者只需看**區間寬度**。

:::

| 層 | $SE$ 中位 | 95% CI 寬中位 | 最小可偵測效果 | 對 0.3pp 的檢定力 |
| :--- | ---: | ---: | ---: | ---: |
| 分組層 | 0.408 | 1.600 pp | **1.143 pp** | **11%** |
| 交易層 | 0.223 | 0.876 pp | **0.626 pp** | 27% |

**分組層的 MDE（1.14 pp）大於 GICS 參照臂自身的等權年化（+0.461%）**
→ 連「效果大到足以翻轉參照臂損益方向」的差異都未必看得見。

> 分組層的 null **幾乎不帶資訊**；但其點估計方向 9/9 一致、量級中位 −0.538 pp，
> 且最負的一格已於校正前顯著——資料無法排除「資料驅動分群顯著較差」，
> 其程度遠甚於無法排除「兩者相當」。
>
> 交易層則相反：MDE 0.63 pp 與觀測中位 +0.579 pp 同量級，
> 效果恰落在看得見的邊界上，故該層能通過校正而分組層不能。

## 分組限制強度：一條有序的維度

各分組方式對候選池施加的限制強度不同。依排除標的比例排列，並併入不分組零點：

| 分組方式 | 排除標的 | 強平率 | 平均 Sharpe | 等權年化 |
| :--- | ---: | ---: | ---: | ---: |
| **不分組** | **0%** | **33.5%** | **+0.109** | **+0.676%** |
| GICS 產業 | 15.1% | 36.1% | +0.062 | +0.461% |
| HDBSCAN | 26.4% | 36.7% | −0.053 | +0.019% |
| Agglomerative | 35.3% | 37.9% | +0.010 | +0.236% |
| K-means | 42.0% | 39.0% | −0.179 | −0.386% |

**逐格觀之**（平均 Sharpe）：

| 分組 | SSD | DTW | SSD-DTW-PCA |
| :--- | ---: | ---: | ---: |
| **不分組** | **+0.110** | **+0.105** | **+0.112** |
| GICS 產業 | +0.043 | +0.068 | +0.076 |
| Agglomerative | −0.040 | +0.028 | +0.042 |
| HDBSCAN | −0.129 | +0.012 | −0.044 |
| K-means | −0.215 | −0.158 | −0.165 |

::: {.callout-important}
**不分組三格全部優於同排序的 GICS；GICS 三格又全部優於任一資料驅動分群格。**

⚠️ 惟兩者最接近處僅差 **0.001** 個 Sharpe 單位（GICS-SSD +0.043 對 AGG-SDP +0.042）——
該分離在現行資料上成立，但已無實質餘裕，不應視為穩健。
:::

> ⚠️ 非嚴格單調：Agglomerative 排除標的多於 HDBSCAN 卻表現較好——已知反例，
> 故稱之為趨勢而非定律。

# 4.2 交易層：門檻選擇 vs. 固定門檻

## 主檢定：五種配對底方向全正，校正後 3/5 顯著

對照的兩臂為 **DL-THR**（逐期由模型選擇進場門檻）與 **Z-Score**（固定 $z$=2.0）。
兩臂共用同一批配對、同一組參數格，唯一變因是門檻如何決定。

| 配對底 | 年化Δ(pp) | 95% CI (pp) | IR | 勝日% | $p$ | BH 校正 $p$ |
| :--- | ---: | :---: | ---: | ---: | ---: | ---: |
| **GICS × SSD（傳統）** | **+0.798** | [+0.17, +1.69] | 0.505 | 51.3 | 0.0403 | 0.0504 |
| **HDBSCAN × SDP** | **+0.666** | [+0.27, +1.15] | 0.493 | 52.1 | 0.0042 | **0.0210** |
| **GICS × SDP（傳統）** | **+0.579** | [+0.17, +1.05] | 0.429 | 51.8 | 0.0111 | **0.0278** |
| **K-means × SSD** | +0.310 | [+0.03, +0.59] | 0.267 | 51.6 | 0.0297 | **0.0495** |
| Agglomerative × SSD | +0.239 | [−0.09, +0.61] | 0.189 | 51.5 | 0.1802 | 0.1802 |

::: {.callout-important}

**方向 5/5 為正，BH 校正後 3/5 顯著**（校正前 4/5）。
這是**全篇唯一通過多重檢定校正的一組**。

觀測效果中位 +0.579 pp 對最小可偵測效果 0.63 pp
→ 效果恰好落在本設計看得見的邊界上。

⚠️ 邊界極不穩固：K-means 以 **0.0495** 通過、GICS-SSD 以 **0.0504** 未通過，
兩者相差 0.0009。**不應以「顯著／不顯著」的二分法陳述本表。**
且通過的是**相對於 Z-Score 的差**——同一批策略的絕對檢定 6/6 不顯著、
DSR 6/6 未過門檻（見 4.6）。

:::

> **增益最大者為兩條傳統 GICS 配對底** → 門檻選擇的改良與「配對如何找到」**正交**，
> 不依賴資料驅動分群，在傳統產業分組上反而取得最大增益。

::: {.aside}
五輪獨立重訓的跨輪標準差中位 0.031、全距中位 0.069，小於增益本身。對沖口徑修正造成的位移已由預先註冊的判準確認超出重訓雜訊（120 格中 88 格，$p$ = 3.7e−36）；唯 `KM-SSD-DRL` 臂 6/15（$p$ = 0.061）不可單獨宣稱。
:::

## 單一參數配置的檢定力限制

等權組合的解析度來自平均掉各配置的特異噪音。逐格檢定：

| 配對底 | 15 格中正向顯著 | 年化Δ 中位 (pp) | 年化Δ 全距 (pp) |
| :--- | :---: | ---: | :---: |
| HDBSCAN × SDP | 6/15 | +0.647 | −0.43 ~ +1.95 |
| GICS × SDP | 5/15 | +0.532 | −0.97 ~ +2.37 |
| GICS × SSD | 5/15 | +0.759 | −0.59 ~ +3.14 |
| K-means × SSD | 4/15 | +0.404 | −0.17 ~ +0.88 |
| Agglomerative × SSD | 3/15 | +0.231 | −0.37 ~ +0.95 |

**單一參數配置的檢定力顯著低於等權組合**——全距普遍橫跨零。

> 此即 3.5.5 節採等權口徑的理由之一：
> 逐格報告不僅檢定力低，且「挑最好的那一格」會引入選擇偏誤。

## 增益來源①：不是拉高門檻

DL-THR 實際進場門檻中位數 **2.19–2.37**，高於基準 2.0。
若增益僅來自「調高門檻」，把 Z-Score 拉到同一水準應能複製。

| 配對底 | A：DL−ZS(2.0) | B：DL−ZS(同門檻) | C：門檻管道 | **複製率** |
| :--- | ---: | ---: | ---: | ---: |
| GICS-SSD | +0.798 | +0.676 ~ +0.769 | +0.029 ~ +0.122 | **3.6 ~ 15.3%** |
| HDBSCAN | +0.666 | +0.576 ~ +0.611 | +0.055 ~ +0.090 | 8.3 ~ 13.5% |
| GICS-SDP | +0.579 | +0.537 ~ +0.594 | −0.015 ~ +0.042 | −2.6 ~ 7.3% |
| K-means | +0.310 | +0.210 ~ +0.250 | +0.060 ~ +0.100 | 19.4 ~ 32.3% |
| Agglomerative | +0.239 | +0.061 ~ +0.149 | +0.091 ~ +0.178 | **38.1 ~ 74.5%** |

**門檻管道在四個配對底僅能複製表面增益的 −3 ~ 32%**，且對照 B 的效果量
幾乎未縮水（GICS-SSD 自 +0.798 僅降至 +0.676）。

> 增益不是「單純調高門檻」的效果——**但 Agglomerative 是例外**：
> 其複製率 38–75%，且純門檻管道 ZS(2.2)−ZS(2.0) 自身即達 5% 顯著
> （+0.178 pp，$p$ = 0.019），而該底的 A 對照本就不顯著。
> 該底那點微弱的表面增益，多半就是調高門檻的效果。

## 增益來源②：不是 SKIP 的選股方向

DL-THR 的 SKIP 率為 **30.1–32.2%**（逐格平均）。

::: {.callout-important}

底層策略期望值為負 → **隨機跳過任一批配對，期望上都會「避開損失」**
→ 必須以置換檢定建立虛無分布（每格 2,000 次不放回重抽）。

:::

| 配對底 | 實際避損 | 隨機期望 | 技巧成分 | 平均百分位 | 顯著格數 |
| :--- | ---: | ---: | ---: | ---: | :---: |
| Agglomerative | −84.5 | +17.4 | **−101.9** | 43.0 | 0/15 |
| K-means | +138.9 | +359.0 | **−220.1** | 30.7 | 0/15 |
| GICS-SDP | +13.9 | −355.2 | +369.1 | 68.3 | 2/15 |
| HDBSCAN | +343.0 | +17.7 | +325.4 | 72.7 | 3/15 |
| GICS-SSD | +203.6 | −270.8 | +474.4 | 81.2 | 3/15 |

**75 格中 8 格顯著（隨機期望 3.75，單尾二項 $p$ = 0.034）**
→ 拒絕虛無假設，即 **SKIP 確有選股技巧**，「避損純屬機械效應」被排除。

⚠️ 舊版報 4/75（$p$ = 0.52）並據以寫「不具可證實的技巧」，
該敘述已撤回——舊版讀到的是未失效的配對期快取（附錄 B.5.4）。

惟證據弱且集中於 GICS-SSD／HDBSCAN／GICS-SDP 三底；
Agglomerative 與 K-means 的技巧成分為**負**（比亂跳更差）。

## 增益來源③：也不是總曝險減少

前一項對齊的是**門檻**而非**曝險**：即使門檻拉到 2.3，
Z-Score 的進場次數仍為 3,115–3,340，而 DL-THR 為 2,107–2,464。

故另建虛無分布——**固定門檻 2.0，但隨機跳過與 DL-THR 同數量的配對期**：

| 配對底 | ZS 全額 | 隨機跳過 | DL-THR 實際 | 超額 | 百分位 | 顯著格數 |
| :--- | ---: | ---: | ---: | ---: | ---: | :---: |
| **GICS-SSD** | +1,072 | +801 | **+3,063** | **+2,262** | 92.5 | **11/15** |
| **HDBSCAN** | +341 | +359 | **+2,002** | **+1,643** | 89.1 | **11/15** |
| **GICS-SDP** | +1,682 | +1,327 | **+3,126** | **+1,799** | 86.8 | **9/15** |
| K-means | −1,125 | −766 | −351 | +415 | 72.7 | 7/15 |
| Agglomerative | +233 | +250 | +830 | +580 | 73.5 | 6/15 |

> 兩條 GICS 底的 **ZS 全額高於隨機跳過**（+1,072 > +801、+1,682 > +1,327）——
> 對期望值為正的底層策略，少交易本身是虧的，
> 故「降低曝險」不可能是其增益來源。
> 另三底則相反，本項只證明「DL-THR 贏過隨機少做」。

::: {.callout-note}
本表的顯著性於 2026-09-08 更正：先前沿用替代解釋一的判定欄，
但那問的是「跳得比隨機好嗎」，而本項問的是
「DL-THR 的**總損益**贏過『同樣少交易但隨機挑』嗎」——兩者檢定量不同。
:::

::: {.aside}
三項檢定共同侷限：皆為事後重抽，不含槽位再配置效應。
:::

## 4.2.4 反事實標籤的價值：被自身資料否證的預期

§3.4.4 指出本交易端的**全資訊**性質——9 個動作的報酬皆可精確反事實回算。
該性質原被**預期**為其樣本效率之來源。

**RL-THR** 保持動作選單、狀態、網路與 walk-forward 切分**逐位元相同**，
僅將訓練標籤縮成「實際選中的那一個」並改採 $\varepsilon$-greedy 探索。
配對底 `Grid (AGG-SSD)`，15 格等權、逐日差分、循環 block bootstrap。

| 對照 | 年化Δ(pp) | 95% CI (pp) | $p$ |
| :--- | ---: | :---: | ---: |
| DL-THR − Z-Score | +0.239 | [−0.09, +0.61] | 0.180 |
| **RL-THR($\varepsilon$=0.05) − Z-Score** | **+0.528** | [+0.01, +1.28] | 0.092 |
| **RL-THR($\varepsilon$=0.10) − Z-Score** | **+0.479** | [−0.03, +1.23] | 0.130 |
| RL-THR($\varepsilon$=0.05) − DL-THR | +0.288 | [−0.13, +0.83] | 0.231 |
| RL-THR($\varepsilon$=0.10) − DL-THR | +0.240 | [−0.18, +0.78] | 0.326 |

::: {.callout-important}
**三個部分回饋變體的點估計全部優於全資訊版本**，但差異**不顯著**。

可陳述者僅為否定的一半：
**資料不支持「全資訊標籤是本交易端效果之來源」這一預期。**
:::

> **命名不因此改變**——依據是學習問題的**性質**（動作報酬可完整反事實回算），
> 該性質為事實；受否證者是「該性質帶來優勢」這一**預期**。

# 4.3 組合系統：實務上要部署的那個檢定

4.1 與 4.2 各測一個成分，**都不是實務上要部署的系統**。

> **組合系統**　資料驅動分群 + 排序 + 篩選 + **DL-THR 交易端**
>
> **傳統基準**　GICS 產業分組 + 同一排序 + 同一篩選 + **固定門檻 Z-Score**

**全期（2001–2025）**

| 分群法 | 傳統基準 | 年化Δ(pp) | IR | 95% CI (pp) | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | ---: | :---: | ---: | ---: |
| Agglomerative | GICS-SSD | **−0.097** | −0.037 | [−1.00, +0.74] | 0.831 | 0.831 |
| HDBSCAN | GICS-SDP | +0.128 | +0.050 | [−0.70, +0.93] | 0.760 | 0.831 |
| K-means | GICS-SSD | **−0.570** | −0.208 | [−1.56, +0.34] | 0.232 | 0.695 |

::: {.callout-important}
**三組中兩負一正，校正後無一顯著。**
對沖口徑修正後 HDBSCAN 那組已由負轉正，故**不再宣稱「完整系統劣於傳統基準」**；
可陳述者僅為「三組皆不顯著，點估計散布於零的兩側」——原因見下頁的成分分解。
:::

## 組合系統（續）：2012 年後

排除 2008 金融海嘯與其後的高波動期後重跑同一組對照：

| 分群法 | 傳統基準 | 年化Δ(pp) | IR | 95% CI (pp) | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | ---: | :---: | ---: | ---: |
| Agglomerative | GICS-SSD | +0.470 | 0.259 | [−0.36, +1.21] | 0.234 | 0.350 |
| HDBSCAN | GICS-SDP | +0.193 | 0.110 | [−0.53, +0.84] | 0.587 | 0.587 |
| K-means | GICS-SSD | +0.582 | 0.306 | [−0.21, +1.31] | 0.135 | 0.350 |

**方向翻正，但仍無一顯著。**

> 全期為負、2012 後為正，差異來自 2008–2011 這段。
> 該期間的分組層損害最大——與 4.6.1 的 regime 分層一致：
> 動盪期是策略唯一獲利的環境，而分組限制在該環境的代價也最高。

## 成分分解：損害來自哪一半？

$$(\text{分群}+\text{DL})-(\text{GICS}+\text{ZS}) = \underbrace{(\text{分群}+\text{DL})-(\text{分群}+\text{ZS})}_{\text{DL-THR 成分}} + \underbrace{(\text{分群}+\text{ZS})-(\text{GICS}+\text{ZS})}_{\text{分群成分}}$$

| 期間 | 分群法 | 總效果 | DL-THR 成分 | $p$ | 分群成分 | $p$ |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: |
| 全期 | Agglomerative | −0.097 | **+0.239** | 0.180 | **−0.336** | 0.472 |
| 全期 | HDBSCAN | +0.128 | **+0.666** | 0.004 | **−0.538** | 0.139 |
| 全期 | K-means | −0.570 | **+0.310** | 0.030 | **−0.881** | 0.066 |
| 2012+ | Agglomerative | +0.798 | +0.278 | 0.108 | +0.521 | 0.157 |
| 2012+ | HDBSCAN | +0.527 | +0.695 | 0.000 | −0.169 | 0.624 |
| 2012+ | K-means | +0.686 | +0.329 | 0.032 | +0.357 | 0.360 |

::: {.callout-important}
**全期三組的 DL-THR 成分皆正、分群成分皆負。**

在 Agglomerative 與 K-means 兩組，分群層的損害超過交易端的貢獻而淨效果為負；
**HDBSCAN 組相反**（DL +0.666 對分群 −0.538），淨效果為正。
兩成分方向相反且大致相抵——不宜宣稱組合系統優於或劣於基準。
:::

（分解為恆等式，最大殘差 0.001 pp。）

# 4.4 分組層為何是淨損害：期末強制平倉機制

## 損益結構：兩股對衝流量的殘差

399 個 Z-Score 基準格的逐筆交易統計（計數器修正後，見附錄 B.2）：

| 指標 | 中位數 |
| :--- | ---: |
| 逐筆勝率 | **0.591** |
| 獲利因子 | 0.995 |
| 期末強制平倉 ÷ 進場 | **0.403** |

**多數交易確實收斂獲利，但獲利被兩股虧損吃到幾乎不剩。**

損益是**三股**流量的殘差（每格平均，初始資金 10,000）：

| 分組方式 | 收斂獲利 | 期末強平 | 停損 | 淨額 | 淨額÷收斂 |
| :--- | ---: | ---: | ---: | ---: | ---: |
| **不分組** | +21,157 | −11,454 | −7,438 | **+2,266** | **10.7%** |
| GICS 產業 | +17,137 | −9,679 | −6,026 | +1,431 | 8.4% |
| HDBSCAN | +16,227 | −9,721 | −6,314 | +192 | 1.2% |
| Agglomerative | +15,400 | −8,997 | −5,638 | +764 | 5.0% |
| K-means | +13,049 | −8,265 | −5,679 | **−895** | **−6.9%** |

**兩股虧損合計吃掉收斂獲利的 89%–107%**；K-means 底的淨額已為負。
以 GICS-SSD 為例：+16,217 / −9,369 / −5,776 → 淨額僅 **+1,072**。

::: {.callout-warning}
**本表於 2026-09-08 重算，並更正一項結構性遺漏。**
先前版本只清點 `EXIT` 與 `PERIOD_END_EXIT`，**完全漏掉 `STOP_LOSS_TRIGGERED`**，
把 GICS-SSD 的淨額算成 +6,775（真值 +1,072，高估約六倍）。
現行淨額與 4.2.3 曝險表的「ZS 全額 +1,072」逐位相符。
:::

> 典型個案（Top 1，第一期 UNH／CI，$\beta$=0.9638）：
> $z$=2.06 進場 → 價差擴大至 $z$=5.46 → 95 日後期末強平，單筆虧 10.9%。

## 強平率與績效：15 個配置一致為負

期末強平率隨分組限制強度**單調上升**，且該序**無**績效序的反例：

| 分組方式 | 排除標的 | 強平率 | 平均 Sharpe |
| :--- | ---: | ---: | ---: |
| **不分組** | **0%** | **33.5%** | **+0.109** |
| GICS 產業 | 15.1% | 36.1% | +0.062 |
| HDBSCAN | 26.4% | 36.7% | −0.053 |
| Agglomerative | 35.3% | 37.9% | +0.010 |
| K-means | 42.0% | 39.0% | −0.179 |

| 層級 | 相關 |
| :--- | :--- |
| 逐臂（15 臂） | Pearson $r$ = **−0.819**（$p$=0.0002） |
| 逐配置內、跨 15 臂 | **15/15 為負**，中位 $r$ = −0.747 |

::: {.callout-warning}
**全樣本混合（n=225）的相關為 +0.483，符號相反**——停損維度造成的 Simpson 悖論
（SL5% 強平率 23.9%／Sharpe −0.233；SL0% 45.0%／+0.183）。**不可引用混合值。**
:::

## 強平的配對只是「尚未回歸」嗎？

取主軸 15 臂的 `Top1/SL0%` 格（無停損，強平不被停損截斷），
對其 **3,107 筆**期末強制平倉的交易往後追蹤 126 個交易日（一個完整交易期），
以引擎自身的出場條件（$z$ 觸及 0）判定回歸：

| 追蹤期 | 累計回歸比例 |
| :--- | ---: |
| 21 日 | 17.5% |
| 42 日 | 25.7% |
| 63 日 | 33.2% |
| **126 日** | **45.4%** |

平倉時 $|z|$ 中位 **3.51**。再給一個完整交易期，**僅 45.4% 回歸**；
未回歸的 **54.6%**，其 $|z|$ 自 4.57 **繼續擴大至 6.82**。

::: {.callout-important}
**這些配對不是尚未回歸，是持續發散。**

延長交易期會讓四成五轉盈、五成五擴大虧損 → 已否證（附錄 C.3）。
且「回歸」只要求 $z$ 曾觸及 0、不計能否成交，故已是對延長交易期有利的上界。
:::

> **機制鏈**：限制候選池 → 選中的配對出樣本收斂性差 → 期末強平實現大額虧損
> → 吃掉收斂交易的獲利。**分組有害不是因為分錯，而是因為縮小了選擇空間。**

# 4.5 兩層改良的檢定力不對等

以主檢定的信賴區間反推標準誤（$SE = $ 區間寬 $/\,3.92$）：

| 層 | 對照組數 | $\lvert\Delta\rvert$ 中位 | $SE$ 中位 | **MDE** |
| :--- | ---: | ---: | ---: | ---: |
| 形成期（分組層） | 9 | 0.538 pp | 0.408 | **1.143 pp** |
| 交易期（交易層） | 5 | 0.579 pp | 0.223 | **0.626 pp** |

**形成期層的 $SE$ 是交易期層的 1.83 倍** → 等效需 **3.3 倍**樣本期間。

**成因是設計，不是資料量：**

| 層 | 兩臂的配對 | 差分消去了什麼 |
| :--- | :--- | :--- |
| 交易期 | **共用同一批** | 市場衝擊 + 配對特異變異 |
| 形成期 | **不同集合** | 僅市場衝擊；標的特異變異殘留 |

> 欲把 MDE 壓到 0.3 pp：形成期層需 **14.5 倍**樣本（逾三個半世紀的日資料），
> 交易期層需 4.4 倍。**更換演算法、增加特徵、改良插補皆不影響此限制。**
> 唯一出路是構造使兩臂共用同一批標的的配對設計。

## 篩選層與分組層的交互作用

文獻多將「分群 + 共整合篩選」並用，未討論兩者的交互作用。排序固定 SSD：

| 分組 | 統計篩選 | 產業 one-hot | 平均 Sharpe | 等權年化 |
| :--- | :---: | :---: | ---: | ---: |
| 不分組 | ADF | — | **+0.110** | **+0.641%** |
| 不分組 | 無 | — | +0.066 | +0.306% |
| Agglomerative | ADF | 1.0 | −0.040 | +0.040% |
| Agglomerative | 無 | 1.0 | −0.033 | −0.094% |
| Agglomerative | ADF | 0 | −0.133 | −0.140% |
| Agglomerative | 無 | 0 | −0.001 | **+0.180%** |

::: {.callout-important}
**篩選層的價值取決於候選池規模。**

不分組時施加篩選：+0.306% → **+0.641%**（有益）
分群且無產業先驗時施加篩選：+0.180% → **−0.140%**（有害）
:::

> 四項條件全開（分群 + 篩選 + 產業先驗，即多數文獻的預設）為 +0.040%，
> 低於不分組的任一格。

::: {.aside}
本頁為描述性比較，未施加統計檢定；最大差距 0.78 pp 低於 4.5 節量化的 1.14 pp 偵測門檻。對應的正式檢定（`prop1_mechanism_contrasts.csv`）14 組經 BH 校正後無一顯著。
:::

# 4.6 風險評估

## Regime 分層

口徑同全章：**15 格等權組合**，且**逐格對齊兩臂**
→ Z-Score 與 DL-THR 之間的唯一變因仍是交易端。

| 配對底 | 交易端 | Calm | Normal | Turbulent |
| :--- | :--- | ---: | ---: | ---: |
| GICS-SSD | Z-Score | −0.42 | −0.41 | **+0.81** |
| GICS-SSD | DL-THR | −0.22 | −0.24 | **+0.83** |
| GICS-SDP | Z-Score | −0.27 | −0.34 | **+0.80** |
| GICS-SDP | DL-THR | 0.00 | −0.16 | **+0.79** |
| HDBSCAN | Z-Score | −0.47 | −0.50 | +0.64 |
| HDBSCAN | DL-THR | −0.20 | −0.13 | +0.62 |
| Agglomerative | Z-Score | −0.39 | −0.44 | +0.56 |
| Agglomerative | DL-THR | −0.14 | −0.22 | +0.41 |
| K-means | Z-Score | −0.42 | −0.91 | +0.39 |
| K-means | DL-THR | −0.07 | −0.60 | +0.25 |

**十列型態一致：動盪期為正，平靜期與一般期為負或零。**
DL-THR 30 格中改善 22 格，集中於虧損較大的兩檔。

::: {.callout-warning}
**此型態不支持「僅於高波動期交易」的策略**——動盪期報酬的 54–87%
集中於 2008–2009 兩年；2022 年動盪日數最多卻近乎零報酬（附錄 C.1）。
:::

## 交易成本敏感度：可行性取決於配對底

成本模型可解析求解：進出場費用 = friction × 名目額，且名目額恰等於每配對資金。
口徑同上頁：15 格等權、逐格對齊兩臂。

| 配對底 | Z-Score 往返 BE% | DL-THR 往返 BE% | Z 餘裕 | DL 餘裕 |
| :--- | ---: | ---: | ---: | ---: |
| GICS-SDP | 0.745 | 0.993 | +16.5 bps | **+41.3 bps** |
| GICS-SSD | 0.691 | 1.010 | +11.1 bps | **+43.0 bps** |
| HDBSCAN | 0.617 | 0.889 | +3.7 bps | +30.9 bps |
| Agglomerative | 0.607 | 0.727 | +2.7 bps | +14.7 bps |
| K-means | 0.428 | 0.511 | −15.2 bps | −6.9 bps |

現行成本假設為往返 **0.58%**（單邊 29 bps）。

::: {.callout-important}
**五個配對底的 DL-THR 皆提高 break-even**（8.3–31.9 bps），
餘裕全距 **−15.2 ~ +43.0 bps**——橫跨 58 bps，恰與成本假設本身同量級。

僅 K-means 底的兩臂餘裕皆為負；另三個分群底的 Z-Score 臂雖已轉正，
但只有 +2.7 ~ +3.7 bps，實務上等同打平。
兩條 GICS 底加 DL-THR 則達 0.99–1.01%，對 0.58% 有逾七成餘裕。
**成本可行性不是全篇一致的結論，而取決於配對底。**

⚠️ 此為對沖口徑修正影響最大的一頁：修正前 AGG 與 HDB 的 Z-Score 臂餘裕為負，
據此曾寫下「三個分群底皆不具可行性」，該敘述已撤回（附錄 B.5）。
:::

## 絕對績效：全部低於無風險利率

依 3.5.5 節口徑，以 15 個參數配置的等權組合報告。主軸 15 格等權年化：

| 分組 | SSD | DTW | SSD-DTW-PCA |
| :--- | ---: | ---: | ---: |
| **不分組** | +0.641% | **+0.780%** | +0.607% |
| GICS 產業 | +0.345% | +0.499% | +0.541% |
| Agglomerative | +0.040% | +0.358% | +0.310% |
| HDBSCAN | −0.287% | +0.269% | +0.076% |
| K-means | −0.484% | −0.330% | −0.343% |

疊加 DL-THR 後最佳者為 **GICS-SSD**：年化 +1.026%、25 年終值 **13,062**、
動用資本口徑 +2.793%、平均 Sharpe 0.276（GICS-SDP 幾乎相同）。

::: {.callout-important}
**同期僅持有無風險資產（2% 假設）將得約 16,400。**

本研究的任何配置皆未達此水準。
:::

**Deflated Sharpe**（$N$=53、$SR_0$=0.408）：不分組×DTW 為 **0.738**，
**無一通過 0.95**。命題 2 的六個主角 DSR 更低（0.036–0.260）。

全宇宙最高者為形成窗 504 對照臂 `GICS-SSD-FW504` 的 **0.885**——
但它後半期 Sharpe 為負：以全期 Sharpe 挑選即會選中它。
**DSR 校正「看過幾個候選」，不校正「以全期指標挑到已失效的策略」。**

另：對「日均報酬是否為零」的絕對檢定 **6/6 不顯著**（$p$ = 0.186–0.850）。

## 相對比較與絕對績效為何是兩件事

| | 逐日差分 bootstrap | 絕對 bootstrap |
| :--- | :--- | :--- |
| $H_0$ | 兩臂績效相同 | 策略平均日報酬為零 |
| 對照物 | 同配對、同參數格的另一臂 | **零** |
| 主張性質 | **相對** | **絕對** |

配對設計消去共同的市場風險 → 訊噪比大幅提高；
絕對檢定沒有對照物可消噪，其標準誤必然大得多。

::: {.callout-important}
本章的三項檢定**全部是相對比較**。交易層確實有一項達到校正後顯著（3/5），
但那只代表「A 優於 B」，不代表 A 本身可獲利。

而本研究的絕對績效**低於無風險利率且 6/6 不顯著**、DSR 6/6 未過門檻，
故連「相對較優者是否值得部署」都不成立。
**這正是相對顯著與絕對可交易必須分開陳述的理由。**
:::

> 這使本研究的定位清楚：**方法論研究**，
> 回答「哪一層的改良可被驗證、哪一層不能」，
> 而非提出一個可交易的策略。

## 本章小結

| 檢定 | 內容 | 方向 | 校正後顯著 |
| :--- | :--- | :--- | :---: |
| 分組層 | 資料驅動分群 vs GICS | 9/9 偏向 GICS | **0/9** |
| 交易層 | DL-THR vs 固定門檻 | 5/5 為正 | **3/5** |
| 兩層組合 | 完整系統 vs 傳統基準 | 全期 2/3 為負 | **0/3** |

**交易層是全篇唯一通過多重檢定校正的一組**；方向的一致性仍是最可靠的訊號。

::: {.callout-important}

**其一，分組層是淨損害**，且限制愈強損害愈大——
不分組 > GICS > 任一資料驅動分群，15 格逐格成立（惟最接近處僅差 0.001）。
機制為期末強制平倉：強平率自 33.5% 升至 39.0%，
與績效的相關在 15 個參數配置中一致為負（中位 $r$ = −0.747）。
強平的配對再追一個交易期僅 45.4% 回歸——**是持續發散，非尚未回歸**。

**其二，門檻選擇是本研究唯一通過多重檢定校正的改良**，且價值與分群無關：
5/5 方向為正、校正後 3/5 顯著，增益最大者為傳統 GICS 底，
三項機械性替代解釋在 GICS-SSD／HDBSCAN／GICS-SDP 三底全排除。
惟「統計顯著」與「機制證據」兩份清單的交集僅 **HDBSCAN 與 GICS-SDP**，
通過與否又落在門檻上（0.0495 對 0.0504），且絕對績效與 DSR 皆未過關。

**其三，兩層的檢定力不對等來自設計**：
形成期層的 $SE$ 為交易期層的 1.83 倍，等效需 3.3 倍樣本。
此為結構性限制，更換演算法或增加特徵皆不影響。

:::

::: {.callout-warning}
**絕對績效**：最佳配置（GICS-SSD + DL-THR）25 年將 10,000 變為 **13,062**，
低於同期無風險利率假設的約 **16,400**；
絕對檢定 6/6 不顯著、DSR 無一通過 0.95。

**本研究不主張任何配置具可交易的獲利能力；可宣稱者全為相對比較。**
:::